# v0.26.0 — Time travel with `as_of()`

## 1. Intro

SurrealDB can keep every version of every record. Since v0.26.0 a QuerySet reads a model's records **as
they were at a moment**: `as_of()` adds SurrealQL's `VERSION` clause to each of its reads, with the moment
always a bound parameter. The returned instances are read-only **snapshots**.

History needs a store that keeps it: the embedded engine's `surrealkv+versioned://` keeps it on disk with
no server, SurrealDB 3.x keeps it when started with `?versioned=true`, SurrealDB 2.x with
`surrealkv+versioned://`. On 3.x a store without history refuses the read; a 2.x server in memory answers
without an error (section 9).

Every section but the last runs without a server.

## 2. Setup

A `surrealkv+versioned://<path>` database on the embedded engine, in a temporary directory. An embedded URL
needs only a namespace and a database; nothing opens until the first ORM call.

In [1]:
import contextlib
import os
import shutil
import tempfile
from datetime import datetime, timedelta

from surreal_orm_lite import SurrealDBConnectionManager

path = tempfile.mkdtemp()
VERSIONED_URL = f"surrealkv+versioned://{path}/shop"
SurrealDBConnectionManager.set_connection(url=VERSIONED_URL, namespace="examples", database="examples")
print("Connection configured:", SurrealDBConnectionManager.is_connection_set())
print("Embedded engine:      ", SurrealDBConnectionManager.is_embedded())

Connection configured: True
Embedded engine:       True


## 3. Model

A plain ORM model: nothing marks it as versioned. History is a property of the store, and `as_of()` a
property of the query.

In [2]:
from surreal_orm_lite import BaseSurrealModel, SurrealConfigDict


class Product(BaseSurrealModel):
    model_config = SurrealConfigDict(table_name="product")

    id: str | None = None
    name: str
    price: float
    stock: int = 0


def show(products: list[Product]) -> list[tuple[str, float, int]]:
    """(name, price, stock) of each product, in order."""
    return [(product.name, product.price, product.stock) for product in products]


print("table:", Product.get_table_name())

table: product


## 4. A short history, written by the ORM

Plain ORM writes. After each step, `RETURN time::now()` takes a moment from the **database's** clock, the
clock its versions are stamped with, and keeps it under a label: the notebook never prints a moment, only
its label.

The `now()` helper asks the manager for the client on every call instead of keeping it in a variable:
section 8 switches to another URL and back, and a client kept from the first engine would still be a
running database on the same path.

In [3]:
async def now() -> datetime:
    """The database's clock: a moment between two writes."""
    client = await SurrealDBConnectionManager.get_client()
    return await client.query("RETURN time::now();")


moments: dict[str, datetime] = {}

lamp = await Product(id="lamp", name="Lamp", price=40.0, stock=12).save()
await Product(id="desk", name="Desk", price=250.0, stock=3).save()
moments["opening"] = await now()  # the lamp and the desk, at their first prices

await lamp.merge(price=30.0)
moments["sale"] = await now()  # the lamp is on sale

desk = await Product.objects().get("desk")
await desk.delete()
moments["now"] = await now()  # the desk is gone

print("moments:", ", ".join(moments))
print("current:", show(await Product.objects().order_by("name").exec()))

moments: opening, sale, now
current: [('Lamp', 30.0, 12)]


## 5. Reading the past

Every read of the queryset honours the moment: `exec()`, `first()`, `get()`, `all()`, `exists()`,
`count()` and the other aggregates, `values()`+`annotate()`, with `filter()`, `order_by()`, `limit()`…

In [4]:
opening = moments["opening"]
print("at the opening:", show(await Product.objects().as_of(opening).order_by("name").exec()))
print("on sale:       ", show(await Product.objects().as_of(moments["sale"]).filter(price__lt=100).exec()))

at the opening: [('Desk', 250.0, 3), ('Lamp', 40.0, 12)]
on sale:        [('Lamp', 30.0, 12)]


A record deleted since is still found at a moment when it existed; `get()` without `as_of()` reads the
current state, where it is gone.

In [5]:
from surreal_orm_lite import SurrealDbNotFoundError

old_desk = await Product.objects().as_of(moments["sale"]).get("desk")
print("the desk on sale:", (old_desk.name, old_desk.price, old_desk.stock))
try:
    await Product.objects().get("desk")
except SurrealDbNotFoundError as error:
    print("the desk now:    ", type(error).__name__)

the desk on sale: ('Desk', 250.0, 3)
the desk now:     SurrealDbNotFoundError


Counts and aggregates at each moment. `values()` with no field groups every record into one row
(`GROUP ALL`), so `annotate()` sums up the whole shop.

In [6]:
from surreal_orm_lite import Avg, Count, Sum

for label, moment in moments.items():
    at = Product.objects().as_of(moment)
    count = await at.count()
    [shop] = await at.values().annotate(products=Count(), units=Sum("stock"), mean_price=Avg("price")).exec()
    print(
        f"{label + ':':<8} count() = {count} | products = {shop['products']}, units = {shop['units']}, "
        f"mean price = {shop['mean_price']}"
    )

opening: count() = 2 | products = 2, units = 15, mean price = 145.0
sale:    count() = 2 | products = 2, units = 15, mean price = 140.0
now:     count() = 1 | products = 1, units = 12, mean price = 30.0


## 6. Other ways to give the moment

Besides a `datetime` (naive means UTC) and an ISO-8601 string, `as_of()` takes "how long ago", a
`timedelta` or an ORM `Duration` subtracted from the database's clock, and a `SurrealFunc` the database
evaluates. Both are resolved by the database just before each read. `None` reads the current state again.

The shop was created while this notebook ran, so an hour ago it held nothing, on any date.

In [7]:
from surreal_orm_lite import SurrealFunc

print("as_of(timedelta(0)):                   ", await Product.objects().as_of(timedelta(0)).count())
print('as_of(SurrealFunc("time::now() - 1h")):', await Product.objects().as_of(SurrealFunc("time::now() - 1h")).count())
print("as_of(opening).as_of(None):            ", await Product.objects().as_of(opening).as_of(None).count())

as_of(timedelta(0)):                    1
as_of(SurrealFunc("time::now() - 1h")): 0
as_of(opening).as_of(None):             1


A version is stamped in nanoseconds since 1970 in a signed 64-bit integer, so the moment must lie between
1970-01-01T00:00:00Z and 2262-04-11T23:47:16.854775Z. SurrealDB 2.x would read an earlier moment as the
**current** state, without an error, and every line fails after the later one, so the ORM refuses a moment
outside that range on every line: an absolute moment at the call (`ValueError`), a duration or an expression
once the database has resolved it, before the read (`SurrealDbError`).

In [8]:
try:
    Product.objects().as_of(datetime(1900, 1, 1))
except ValueError as error:
    print(error)

as_of() takes a moment from 1970-01-01T00:00:00+00:00 to 2262-04-11T23:47:16.854775+00:00, the instants SurrealDB versions can address (SurrealDB 2.x reads an earlier moment as the current state), got 1900-01-01T00:00:00+00:00


## 7. Snapshots are read-only

An instance read with `as_of()` holds the record as it was, not the current record: `save()`, `update()`,
`merge()`, `delete()` and the other writes refuse it before any request. The full message names the moment,
so only its first part is printed here.

In [9]:
from surreal_orm_lite import SurrealDbError

old = await Product.objects().as_of(moments["sale"]).get("desk")
try:
    await old.save()
except SurrealDbError as error:
    print(str(error).split(":")[0])

Product.save() cannot run on an instance read with as_of()


To restore a past state, build a new instance from the snapshot: `save()` creates the deleted desk again
(`update()` would write the past state over a record that still exists). `refresh()` turns a snapshot back
into the current record, and its writes work again.

In [10]:
await Product(**old.model_dump()).save()
print("current:", show(await Product.objects().order_by("name").exec()))

await old.refresh()
await old.merge(stock=5)
print("after refresh() and merge():", show([await Product.objects().get("desk")]))

current: [('Desk', 250.0, 3), ('Lamp', 30.0, 12)]
after refresh() and merge(): [('Desk', 250.0, 5)]


## 8. A store without history

On the embedded engine the URL tells whether the store keeps history: `mem://`, `surrealkv://` and the
other schemes keep none, and `as_of()` raises before any request, naming `surrealkv+versioned://`. Back on
the versioned path, the engine reopens the same directory with its history.

In [11]:
SurrealDBConnectionManager.set_connection(url="mem://", namespace="examples", database="examples")
try:
    await Product.objects().as_of(timedelta(0)).count()
except SurrealDbError as error:
    print(error)

SurrealDBConnectionManager.set_connection(url=VERSIONED_URL, namespace="examples", database="examples")
print()
print("back on the versioned path, at the opening:", show(await Product.objects().as_of(opening).order_by("name").exec()))

as_of() needs a store that keeps history, and the embedded engine at 'mem://' keeps none. Open a surrealkv+versioned://<path> database (the surrealdb SDK 2.x ignores ?versioned=true), or connect to a server started with versioned storage.

back on the versioned path, at the opening: [('Desk', 250.0, 3), ('Lamp', 40.0, 12)]


## 9. On a server (optional)

The same read on a SurrealDB server over WebSocket (`SURREALDB_HOST` and `SURREALDB_PORT`, default
`localhost:8000`). A **3.x** server started without versioned storage refuses `VERSION`, and the ORM raises
that as a `SurrealDbError` (the SDK's error is its `__cause__`). A **2.x** server in memory or on plain
SurrealKV does not refuse it, and the ORM cannot tell which storage it runs: the cell prints that note
instead of reading. Without a server the cell says so and the notebook goes on.

In [12]:
import socket

HOST = os.environ.get("SURREALDB_HOST", "localhost")
PORT = os.environ.get("SURREALDB_PORT", "8000")


async def drop_products() -> None:
    """Drop the product table on the server, whether or not it exists."""
    client = await SurrealDBConnectionManager.get_client()
    with contextlib.suppress(Exception):
        await client.query("REMOVE TABLE product;")


try:
    with socket.create_connection((HOST, int(PORT)), timeout=2):
        pass
    SurrealDBConnectionManager.set_connection(
        url=f"ws://{HOST}:{PORT}/rpc", user="root", password="root", namespace="examples", database="examples"
    )
    version = await (await SurrealDBConnectionManager.get_client()).version()
except Exception:  # nothing listening there, or the server refused the connection
    print("no server reachable")
else:
    line = int(str(version).removeprefix("surrealdb-").split(".")[0])
    print(f"SurrealDB {line}.x")
    if line >= 3:
        await drop_products()
        await Product(id="lamp", name="Lamp", price=40.0, stock=12).save()
        try:
            count = await Product.objects().as_of(timedelta(0)).count()
        except SurrealDbError as error:
            print(str(error).split(". ")[0] + ".")
        else:
            print("this server keeps history; products now:", count)
        await drop_products()
    else:
        print(
            "SurrealDB 2.x in memory or on plain SurrealKV answers a time-travel read without an error, with only "
            "the records last written at or before the moment, in their current state. Start it with "
            "surrealkv+versioned:// for time travel."
        )

SurrealDB 3.x
as_of() needs a store that keeps history, and this SurrealDB server's storage does not (There was a problem with the key-value store: The underlying datastore does not support versioned queries).


## 10. Cleanup

`unset_connection()` clears the settings and closes every connection; then the temporary directory goes.

In [13]:
await SurrealDBConnectionManager.unset_connection()
shutil.rmtree(path)
print("Connection configured:", SurrealDBConnectionManager.is_connection_set())
print("temporary directory removed")

Connection configured: False
temporary directory removed
